# 01 — Data Cleaning & Preprocessing Pipeline
**Project:** CustomerPulse AI — E-commerce Customer Intelligence & Churn Prediction System

### Objective
Audit, clean, and validate the raw UCI Online Retail dataset (`541,909` rows) using explicit, documented business rules:
1. Remove exact duplicate line items (`5,268` rows).
2. Remove bad-debt adjustment invoices (`A` prefix) and warehouse damage/write-off entries (`UnitPrice <= 0` or `Quantity <= 0` without a `'C'` cancellation prefix).
3. Exclude non-product accounting/postage `StockCode`s (`POST`, `DOT`, `M`, `BANK CHARGES`, `D`, `CRUK`, `C2`, `PADS`, `AMAZONFEE`).
4. Standardize product descriptions to their modal (most frequent) text per `StockCode`.
5. Preserve cancellation invoices (`C` prefix) with `is_cancellation = True` so Gross Revenue, Returns/Cancellations, and Net Revenue are accurately tracked.
6. Output both `cleaned_transactions.csv` (store-wide, including guest checkouts) and `customer_transactions.csv` (identified customers for RFM & Churn modeling).

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEANED_TRANSACTIONS_PATH,
    CUSTOMER_TRANSACTIONS_PATH,
    CLEANING_REPORT_JSON_PATH,
)

## 1. Load Stage 3 Cleaning Waterfall Summary

In [ ]:
with open(CLEANING_REPORT_JSON_PATH, "r", encoding="utf-8") as f:
    cleaning_summary = json.load(f)

waterfall_df = pd.DataFrame(
    list(cleaning_summary["row_waterfall"].items()),
    columns=["Pipeline Step", "Row Count"]
)
waterfall_df

## 2. Financial Reconciliation: Store-Wide vs. Identified Customers

In [ ]:
financials_df = pd.DataFrame({
    "Store-Level (All Checkouts)": cleaning_summary["store_level_financials"],
    "Customer-Level (Identified Only)": cleaning_summary["customer_level_financials"]
})
financials_df

## 3. Inspect Processed Customer Transactions Dataset

In [ ]:
df_customers = pd.read_csv(
    CUSTOMER_TRANSACTIONS_PATH,
    dtype={"InvoiceNo": str, "StockCode": str, "CustomerID": str},
    parse_dates=["InvoiceDate"]
)
print(f"Loaded customer_transactions.csv: {df_customers.shape[0]:,} rows, {df_customers.shape[1]} columns")
df_customers.head(10)

## 4. Verify Separation of Qualifying Sales vs. Cancellations/Returns

In [ ]:
tx_breakdown = (
    df_customers.groupby("TransactionType")
    .agg(
        row_count=("InvoiceNo", "count"),
        unique_invoices=("InvoiceNo", "nunique"),
        unique_customers=("CustomerID", "nunique"),
        min_quantity=("Quantity", "min"),
        max_quantity=("Quantity", "max"),
        total_revenue_gbp=("LineTotal", "sum"),
    )
)
tx_breakdown